In [ ]:
from pyspark.sql import SparkSession

In [ ]:
spark = SparkSession.builder \
         .appName("late_streaming_events") \
         .master("local[*]") \
         .config("spark.jars.packages","org.apache.spark:spark-sql-kafka-0-10_2.12:3.1.2") \
         .config("spark.sql.shuffle.partitions", "4") \
         .config("spark.driver.memory", "2g") \
         .enableHiveSupport() \
         .getOrCreate()
spark.sparkContext.setLogLevel("WARN")

Recover late events

In [ ]:
%run ./utils.ipynb

import pyspark.sql.functions as F
from pyspark.sql.types import StructType, StructField,LongType, IntegerType, StringType

db = "looker_ecommerce_dwh"
kafka_bootstrap_servers = "kafka1:19092,kafka2:19093,kafka3:19094"
topic = "events"

lookback_days = 3

recover_after_hours = 2

event_schema = StructType([
    StructField("event_id", LongType(), True),
    StructField("user_id", IntegerType(), True),
    StructField("sequence_number", IntegerType(), True),
    StructField("session_id", StringType(), True),
    StructField("created_at", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("browser", StringType(), True),
    StructField("traffic_source", StringType(), True),
    StructField("uri", StringType(), True),
    StructField("event_type", StringType(), True),
])

Cleaning

In [ ]:
def valid_event():
    """The rule for a VALID event. Needs created_at already converted to a timestamp"""
    return (F.col("event_id").isNotNull()
            & F.col("created_at").isNotNull()
            & (F.col("created_at") <= F.current_timestamp() + F.expr("INTERVAL 1 MINUTE"))
            & (F.col("sequence_number") >= 1))

def clean_events(df):
    """
    Same cleaning rules as silver_layer, applied to every streamed event.
    """
    return (
        df
        .withColumn("created_at", F.to_timestamp("created_at"))
        .withColumn("session_id", F.regexp_replace(F.trim(F.col("session_id")), r"\s+", ""))
        .withColumn("uri", F.regexp_replace(F.trim(F.col("uri")), r"\s+", ""))
        .withColumn(
            "city",
            F.initcap(
                F.trim(
                    F.regexp_replace(
                        F.regexp_replace(F.trim(F.col("city")), r"\s+", " "),
                        r"[^\p{L}\s'-]", ""
                    )
                )
            )
        )
        .withColumn(
            "state",
            F.initcap(
                F.trim(
                    F.regexp_replace(
                        F.regexp_replace(F.trim(F.col("state")), r"\s+", " "),
                        r"[^\p{L}\s'-]", ""
                    )
                )
            )
        )
        .withColumn(
            "browser",
            F.when(F.length(F.trim(F.col("browser"))) == 2, F.upper(F.trim(F.col("browser"))))
            .otherwise(F.initcap(F.regexp_replace(F.trim(F.col("browser")), r"\s+", " ")))
        )
        .withColumn(
            "traffic_source",
            F.initcap(F.regexp_replace(F.trim(F.col("traffic_source")), r"\s+", " "))
        )
        .withColumn(
            "event_type",
            F.initcap(F.regexp_replace(F.trim(F.col("event_type")), r"\s+", " "))
        )
        .fillna({"city": "Unknown"})
        .filter(valid_event())
    )

Read kafka

In [ ]:
raw = spark.read.format("kafka") \
       .option("kafka.bootstrap.servers", kafka_bootstrap_servers) \
       .option("subscribe", topic) \
       .option("startingOffsets", "earliest") \
       .option("endingOffsets", "latest") \
       .load() \
       .filter(F.col("timestamp") >= F.expr(f"current_timestamp() - INTERVAL {lookback_days} DAYS"))

events = (clean_events(raw
                       .select(F.from_json(F.col("value").cast("string"), event_schema).alias("e"))
                       .select("e.*"))
          .filter(F.col("created_at") < F.expr(f"current_timestamp() - INTERVAL {recover_after_hours} HOURS"))
          .dropDuplicates(["event_id"])
          .persist())

n_events = events.count()
print(f"Unique valid events found in Kafka (last {lookback_days} days, older than {recover_after_hours} h): {n_events}")

Enrich (Same logic as streaming layer)

In [ ]:
def enrich(batch_df):
    def date_key(c):
        return F.date_format(c, "yyyyMMdd").cast("int")

    return (attach_user_key(batch_df, "created_at", "event_id", spark, db)
            .select(
                "event_id",
                "user_key",
                date_key("created_at").alias("created_date_key"),
                "session_id",
                "sequence_number",
                "event_type",
                "uri",
                "browser",
                "traffic_source",
                "city",
                "state",
                F.year("created_at").alias("event_year"),
                F.month("created_at").alias("event_month"),
            ))

Insert into fact_events

In [ ]:
if n_events == 0:
    print("Nothing to reconcile.")
else:
    enriched = enrich(events)
    
    simple_load(
        df=enriched,
        table_name=f"{db}.fact_events",
        business_key="event_id",
        partition_cols=["event_year", "event_month"],
        spark=spark,
    )
    
events.unpersist()